In [1]:
"""
Random Forest classification workflow for detecting abandoned peat extraction areas.

This script:
- trains 3×3 and 5×5 Random Forest models,
- evaluates train and validation performance,
- calculates abandoned-class ROC and threshold metrics,
- generates SHAP plots for model interpretation.

Classes:
0 = active peat extraction area
1 = abandoned peat extraction area
2 = restored peat extraction area
"""

import numpy as np
import geopandas as gpd
import shap
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from joblib import dump

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

# -------------------------
# 1) Config
# -------------------------
BASE_DIR = Path("path\to\work\folder\")

RUNS = {
    "peatmasked": {
        "input_gpkg": BASE_DIR / "training_points_with_predictors_all_classes.gpkg", # name of your point file if it's in the work folder
    }
}

WINDOWS = {
    "3x3": ["dem_sd3", "vv_sd3", "vh_sd3", "twi_sd3"],
    "5x5": ["dem_sd5", "vv_sd5", "vh_sd5", "twi_sd5"]
}

base_predictors = [
    "ndvi", "ndwi", "ndmi", "swir1", "swir2",
    "dem", "vv", "vh", "twi"
]

summary_rows = []

plt.rcParams["font.family"] = "DejaVu Sans"  # you can choose the font. This one supports special characters needed in Estonian

# -------------------------
# 2) Hyperparameter search
# -------------------------
def search_hyperparams(X, y, random_state=42):
    """
    Tune RandomForestClassifier with RandomizedSearchCV
    over n_estimators and max_depth.
    """

    n_estimators = list(np.arange(50, 201, 10))
    max_depth = list(np.arange(5, 21, 1)) + [None]

    param_distributions = {
        "n_estimators": n_estimators,
        "max_depth": max_depth
    }

    estimator = RandomForestClassifier(
        random_state=random_state,
        n_jobs=-1,
        class_weight="balanced"
    )

    rf_random = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=30,
        cv=5,
        verbose=2,
        random_state=random_state,
        n_jobs=-1,
        scoring="f1_weighted",
        return_train_score=True
    )

    rf_random.fit(X, y)

    best_params = rf_random.best_params_.copy()
    return best_params, rf_random


# -------------------------
# 3) SHAP helper
# -------------------------
def get_multiclass_shap(rf_model, X_explain, class_index=1):

    explainer = shap.TreeExplainer(rf_model)

    shap_values_raw = explainer.shap_values(X_explain)

    if isinstance(shap_values_raw, list):
        shap_values_class = shap_values_raw[class_index]
        expected_value = explainer.expected_value[class_index]

    elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
        shap_values_class = shap_values_raw[:, :, class_index]
        expected_value = explainer.expected_value[class_index]

    else:
        raise RuntimeError("Unexpected SHAP format")

    shap_explanation = shap.Explanation(
        values=shap_values_class,
        base_values=np.full(X_explain.shape[0], expected_value),
        data=X_explain.values,
        feature_names=list(X_explain.columns)
    )

    return shap_explanation


# -------------------------
# 4) Run all combinations
# -------------------------
for run_name, cfg in RUNS.items():
    print("\n" + "=" * 70)
    print(f"DATASET: {run_name}")
    print("=" * 70)

    gdf = gpd.read_file(cfg["input_gpkg"])

    print("Input file:", cfg["input_gpkg"])
    print("\nAvailable columns:")
    print(sorted(gdf.columns))

    for window_name, focal_predictors in WINDOWS.items():
        print("\n" + "-" * 70)
        print(f"WINDOW: {window_name}")
        print("-" * 70)

        predictors = base_predictors + focal_predictors

        model_path = BASE_DIR / f"rf_classifier_tuned_multiclass_{run_name}_{window_name}.joblib"
        validation_csv = BASE_DIR / f"validation_predictions_multiclass_{window_name}_{run_name}.csv"
        train_csv = BASE_DIR / f"train_predictions_multiclass_{window_name}_{run_name}.csv"
        shap_dir = BASE_DIR / f"shap_outputs_multiclass_{window_name}_{run_name}"
        tuning_csv = BASE_DIR / f"rf_classifier_tuning_results_multiclass_{window_name}_{run_name}.csv"
        shap_dir.mkdir(exist_ok=True)

        print("\nPredictors used:")
        print(predictors)

        # -------------------------
        # 5) Drop missing values
        # -------------------------
        print("\nRows before:", len(gdf))
        gdf_clean = gdf.dropna(subset=predictors + ["label"]).copy()
        print("Rows after:", len(gdf_clean))

        print("\nMissing values after cleaning:")
        print(gdf_clean[predictors + ["label"]].isna().sum())

        print("\nClass balance:")
        print(gdf_clean["label"].value_counts())

        X = gdf_clean[predictors].copy()
        y = gdf_clean["label"].astype(int).copy()

        # -------------------------
        # 6) Train/test split
        # -------------------------
        X_train, X_valid, y_train, y_valid = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        print("\nTraining samples:", len(X_train))
        print("Validation samples:", len(X_valid))

        # -------------------------
        # 7) Hyperparameter tuning
        # -------------------------
        best_params, rf_random = search_hyperparams(X_train, y_train, random_state=42)

        print("\nBest hyperparameters:")
        print(best_params)
        print("Best CV score:", rf_random.best_score_)

        tuning_results = pd.DataFrame(rf_random.cv_results_)
        tuning_results.to_csv(tuning_csv, index=False)
        print("Saved tuning results to:", tuning_csv)

        # -------------------------
        # 8) Train best model
        # -------------------------
        rf_clf = RandomForestClassifier(
            **best_params,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )

        rf_clf.fit(X_train, y_train)
        
        print("Model class order:", rf_clf.classes_)
        
        dump(rf_clf, model_path)
        print("\nModel saved to:", model_path)

        # -------------------------
        # 9) Evaluate train
        # -------------------------
        y_pred_train = rf_clf.predict(X_train)
        y_prob_train = rf_clf.predict_proba(X_train)
        
        accuracy_train = accuracy_score(y_train, y_pred_train)
        precision_train = precision_score(y_train, y_pred_train, average="weighted", zero_division=0)
        recall_train = recall_score(y_train, y_pred_train, average="weighted", zero_division=0)
        f1_train = f1_score(y_train, y_pred_train, average="weighted", zero_division=0)
        auc_train = roc_auc_score(y_train, y_prob_train, multi_class="ovr", average="weighted")

        print("\nTraining metrics:")
        print("Accuracy_train:", accuracy_train)
        print("Precision_train:", precision_train)
        print("Recall_train:", recall_train)
        print("F1_train:", f1_train)
        print("ROC_AUC_train:", auc_train)

        print("\nTrain confusion matrix:")
        print(confusion_matrix(y_train, y_pred_train))

        classes = list(rf_clf.classes_)
        
        active_idx = classes.index(0)
        abandoned_idx = classes.index(1)
        restored_idx = classes.index(2)
        
        train_results = X_train.copy()
        train_results["label"] = y_train.values
        train_results["prediction_class"] = y_pred_train
        train_results["prob_active"] = y_prob_train[:, active_idx]
        train_results["prob_abandoned"] = y_prob_train[:, abandoned_idx]
        train_results["prob_restored"] = y_prob_train[:, restored_idx]
        train_results.to_csv(train_csv, index=False)
        
        print("Saved train predictions to:", train_csv)

        # -------------------------
        # 10) Evaluate validation
        # -------------------------
        y_pred = rf_clf.predict(X_valid)
        y_prob = rf_clf.predict_proba(X_valid)
        
        accuracy_valid = accuracy_score(y_valid, y_pred)
        precision_valid = precision_score(y_valid, y_pred, average="weighted", zero_division=0)
        recall_valid = recall_score(y_valid, y_pred, average="weighted", zero_division=0)
        f1_valid = f1_score(y_valid, y_pred, average="weighted", zero_division=0)
        auc_valid = roc_auc_score(y_valid, y_prob, multi_class="ovr", average="weighted")
        
        print("\nValidation metrics:")
        print("Accuracy_valid:", accuracy_valid)
        print("Precision_weighted_valid:", precision_valid)
        print("Recall_weighted_valid:", recall_valid)
        print("F1_weighted_valid:", f1_valid)
        print("ROC_AUC_weighted_ovr_valid:", auc_valid)
        
        print("\nValidation confusion matrix:")
        print(confusion_matrix(y_valid, y_pred))
        
        print("\nClassification report:")
        print(classification_report(
            y_valid,
            y_pred,
            target_names=["active", "abandoned", "restored"],
            zero_division=0
        ))
        
        print("\nProbability stats:")
        print("Min:", y_prob.min())
        print("Max:", y_prob.max())
        print("Mean:", y_prob.mean())
        
        
        valid_results = X_valid.copy()
        valid_results["label"] = y_valid.values
        valid_results["prediction_class"] = y_pred
        valid_results["prob_active"] = y_prob[:, active_idx]
        valid_results["prob_abandoned"] = y_prob[:, abandoned_idx]
        valid_results["prob_restored"] = y_prob[:, restored_idx]
        valid_results.to_csv(validation_csv, index=False)
        
        print("\nSaved validation predictions to:", validation_csv)
        classes = list(rf_clf.classes_)
        
        active_idx = classes.index(0)
        abandoned_idx = classes.index(1)
        restored_idx = classes.index(2)
        # -------------------------
        # 11) SHAP plots for abandoned class
        # -------------------------
        shap_values = get_multiclass_shap(
            rf_clf,
            X_valid,
            class_index=abandoned_idx
        )
        
        # SHAP bar plot
        plt.close("all")
        ax = shap.plots.bar(shap_values, max_display=len(predictors), show=False)
        ax.set_title(f"SHAP tunnuste olulisus - {window_name}", fontsize=14)
        ax.set_xlabel("Keskmine absoluutne SHAP väärtus", fontsize=11)
        ax.set_ylabel("Tunnus", fontsize=11)
        
        ax.figure.tight_layout()
        ax.figure.savefig(
            shap_dir / f"shap_bar_abandoned_{window_name}_{run_name}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.close(ax.figure)
        
        # SHAP beeswarm plot
        plt.close("all")
        shap.plots.beeswarm(shap_values, max_display=13, show=False)
        plt.title(f"SHAP beeswarm - mahajäetud klass - {window_name}", fontsize=14)
        plt.xlabel("Mõju mahajäetud klassi tõenäosusele", fontsize=11)
        plt.ylabel("Tunnused", fontsize=11)
        
        plt.gcf().tight_layout()
        plt.savefig(
            shap_dir / f"shap_beeswarm_abandoned_{window_name}_{run_name}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.close(plt.gcf())

        # -------------------------
        # 12) ROC curve for abandoned class
        # -------------------------
        classes = list(rf_clf.classes_)
        abandoned_idx = classes.index(1)
        
        y_valid_abandoned = (y_valid == 1).astype(int)
        y_prob_abandoned = y_prob[:, abandoned_idx]
        
        # Conservative threshold selected for final mapping
        chosen_threshold = 0.60
        
        threshold_input_csv = shap_dir / f"test_predictions_for_threshold_{window_name}_{run_name}.csv"

        threshold_df = pd.DataFrame({
            "y_true": y_valid_abandoned,
            "y_score": y_prob_abandoned
        })
        
        threshold_df.to_csv(threshold_input_csv, index=False)
        print("Saved threshold input CSV to:", threshold_input_csv)

        fpr, tpr, roc_thresholds = roc_curve(
            y_valid_abandoned,
            y_prob_abandoned
        )

        auc_value = roc_auc_score(
            y_valid_abandoned,
            y_prob_abandoned
        )

        from sklearn.metrics import precision_recall_curve

        prec, rec, pr_thresholds = precision_recall_curve(
            y_valid_abandoned,
            y_prob_abandoned
        )

        f1_scores = 2 * (prec[:-1] * rec[:-1]) / (
            prec[:-1] + rec[:-1] + 1e-10
        )

        best_idx = np.argmax(f1_scores)
        best_threshold = pr_thresholds[best_idx]

        roc_png = shap_dir / f"roc_curve_classifier_tuned_{window_name}_{run_name}.png"

        plt.close("all")
        plt.figure(figsize=(8, 6))

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"RF mudel (AUC = {auc_value:.4f})"
        )

        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            color="gray",
            alpha=0.7,
            label="Juhuslik klassifitseerija"
        )
        idx_thr = np.argmin(np.abs(roc_thresholds - chosen_threshold))
        fpr_thr = fpr[idx_thr]
        tpr_thr = tpr[idx_thr]

        plt.scatter(fpr_thr, tpr_thr, color="red", s=80, zorder=5)

        plt.annotate(
            f"lävi={chosen_threshold:.2f}",
            (fpr_thr, tpr_thr),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=10,
            color="red"
        )

        plt.xlabel("Valepositiivsete määr (FPR)", fontsize=12)
        plt.ylabel("Tõepositiivsete määr (TPR)", fontsize=12)
        plt.title("ROC-kõver", fontsize=14)

        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        plt.xlim(0, 1)
        plt.ylim(0, 1)

        plt.tight_layout()
        plt.savefig(roc_png, dpi=300, bbox_inches="tight")
        plt.close()

        print("Saved ROC curve to:", roc_png)



        # -------------------------
        # 13) Threshold metrics for abandoned class
        # -------------------------
        thresholds_grid = np.arange(0.00, 1.001, 0.01)
        rows_thr = []

        for thr in thresholds_grid:
            y_pred_thr = (y_prob_abandoned >= thr).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                y_valid_abandoned,
                y_pred_thr,
                labels=[0, 1]
            ).ravel()

            rows_thr.append({
                "threshold": thr,
                "tp": tp,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "precision": precision_score(
                    y_valid_abandoned,
                    y_pred_thr,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_valid_abandoned,
                    y_pred_thr,
                    zero_division=0
                ),
                "f1": f1_score(
                    y_valid_abandoned,
                    y_pred_thr,
                    zero_division=0
                )
            })

        threshold_metrics = pd.DataFrame(rows_thr)
        
        best_f1_row = threshold_metrics.loc[threshold_metrics["f1"].idxmax()]
        
        # Get metrics for the chosen threshold
        chosen_row = threshold_metrics.loc[
            np.isclose(threshold_metrics["threshold"], chosen_threshold)
        ].iloc[0]
        chosen_pred_csv = shap_dir / f"validation_abandoned_threshold_{chosen_threshold:.2f}_{window_name}_{run_name}.csv"

        chosen_pred_df = pd.DataFrame({
            "y_true_abandoned": y_valid_abandoned,
            "p_abandoned": y_prob_abandoned,
            "pred_abandoned_thresholded": (y_prob_abandoned >= chosen_threshold).astype(int)
        })
        
        chosen_pred_df.to_csv(chosen_pred_csv, index=False)
        print("Saved chosen-threshold validation predictions to:", chosen_pred_csv)
        
        # Keep this only for reporting the F1-optimal threshold
        best_threshold = best_f1_row["threshold"]

        threshold_metrics_csv = shap_dir / f"threshold_metrics_f1_{window_name}_{run_name}.csv"
        threshold_summary_txt = shap_dir / f"best_threshold_f1_{window_name}_{run_name}.txt"

        threshold_metrics.to_csv(threshold_metrics_csv, index=False)

        with open(threshold_summary_txt, "w", encoding="utf-8") as f:
            f.write("Best threshold by F1-score\n\n")
            f.write(best_f1_row.to_string())
            f.write("\n\n")
            f.write("Chosen conservative threshold for mapping\n\n")
            f.write(chosen_row.to_string())

        print("Saved threshold metrics to:", threshold_metrics_csv)
        print("Saved threshold summary to:", threshold_summary_txt)
        print("Best F1 threshold:", best_threshold)
        print("Chosen mapping threshold:", chosen_threshold)
        print("Chosen threshold metrics:")
        print(chosen_row)
        # -------------------------
        # 11c) SHAP importance CSV
        # -------------------------
        shap_importance = pd.DataFrame({
            "feature": list(X_valid.columns),
            "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
        }).sort_values("mean_abs_shap", ascending=False)

        shap_importance_csv = shap_dir / f"shap_importance_classifier_tuned_{window_name}_{run_name}.csv"
        shap_importance.to_csv(shap_importance_csv, index=False)

        print("Saved SHAP importance CSV to:", shap_importance_csv)


        # -------------------------
        # 14) SHAP waterfall plot
        # -------------------------
        plt.close("all")

        shap.plots.waterfall(
            shap_values[0],
            max_display=len(shap_values.feature_names),
            show=False
        )

        plt.title("Ühe näite klassifikatsiooni selgitus (SHAP)", fontsize=14)

        plt.gcf().tight_layout()
        plt.savefig(
            shap_dir / f"shap_waterfall_classifier_tuned_sample0_{window_name}_{run_name}.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(plt.gcf())

        print("Saved SHAP waterfall to:", shap_dir)
        summary_rows.append({
            "Dataset": run_name,
            "Window": window_name,
            "Accuracy": accuracy_valid,
            "Precision_weighted": precision_valid,
            "Recall_weighted": recall_valid,
            "F1_weighted": f1_valid,
            "ROC_AUC_weighted_ovr": auc_valid,
            "Abandoned_ROC_AUC": auc_value,
            "Best_F1_threshold_abandoned": best_threshold,
            "Best_F1_abandoned_thresholded": best_f1_row["f1"],
            "Chosen_mapping_threshold": chosen_threshold,
            "Chosen_precision": chosen_row["precision"],
            "Chosen_recall": chosen_row["recall"],
            "Chosen_F1": chosen_row["f1"],
            "Chosen_FP": chosen_row["fp"],
            "Chosen_FN": chosen_row["fn"]
                    })
# -------------------------
# 15) Save comparison table
# -------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv = BASE_DIR / "comparison_3x3_vs_5x5_classifier_tuned_train_valid.csv"
summary_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(summary_df)
print("\nSaved comparison CSV to:", summary_csv)


DATASET: peatmasked
Input file: C:\Users\Egert\Documents\peatland_detection\training_points_with_predictors_all_classes.gpkg

Available columns:
['dem', 'dem_sd3', 'dem_sd5', 'geometry', 'label', 'ndmi', 'ndvi', 'ndwi', 'swir1', 'swir2', 'twi', 'twi_sd3', 'twi_sd5', 'vh', 'vh_sd3', 'vh_sd5', 'vv', 'vv_sd3', 'vv_sd5']

----------------------------------------------------------------------
WINDOW: 3x3
----------------------------------------------------------------------

Predictors used:
['ndvi', 'ndwi', 'ndmi', 'swir1', 'swir2', 'dem', 'vv', 'vh', 'twi', 'dem_sd3', 'vv_sd3', 'vh_sd3', 'twi_sd3']

Rows before: 922
Rows after: 922

Missing values after cleaning:
ndvi       0
ndwi       0
ndmi       0
swir1      0
swir2      0
dem        0
vv         0
vh         0
twi        0
dem_sd3    0
vv_sd3     0
vh_sd3     0
twi_sd3    0
label      0
dtype: int64

Class balance:
label
0    433
2    307
1    182
Name: count, dtype: int64

Training samples: 737
Validation samples: 185
Fitting 5 fol